In [0]:
CATALOG = 'SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDERPAYMENTS_ENRICHED'
     

In [0]:
OP_df = spark.table('SALES.BRONZE.ORDERPAYMENTS')
OP_df.limit(1).display()
     

In [0]:
from pyspark.sql.functions import *

OP_enriched_df = OP_df.select(

    trim(col('order_id')).alias('order_id'),
    col('payment_sequential'),
    upper(trim(col('payment_type'))).alias('payment_type'),
    col('payment_installments').alias('payment_installments'),
    col('payment_value').alias('payment_value'),
    col('file_name'),
    col('file_path'),
    col('load_timestamp')
).filter('order_id IS NOT NULL')

display(OP_enriched_df.limit(1)) 

In [0]:
flagged_df = (OP_enriched_df.select("order_id", "payment_sequential", "payment_type", "payment_installments", "payment_value", "file_name", "file_path","load_timestamp") .withColumn("is_payment_value_positive", col("payment_value") > 0))
display(flagged_df)

In [0]:
flagged_df.createOrReplaceTempView('enriched_orderpayments_temp_vw')

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_orderpayments_temp_vw
  )
  WHERE rn = 1
) src
ON (src.order_id  = tgt.order_id)
WHEN MATCHED THEN
  UPDATE SET
    tgt.order_id = src.order_id,
    tgt.payment_sequential = src.payment_sequential,
    tgt.payment_type = src.payment_type,
    tgt.payment_installments = src.payment_installments,
    tgt.payment_value = src.payment_value,
    tgt.file_name = src.file_name,
    tgt.is_payment_value_positive = src.is_payment_value_positive
    
WHEN NOT MATCHED THEN
  INSERT (
    order_id,
    payment_sequential,
    payment_type,
    payment_installments,
    payment_value,
    file_name,
    origin_file_path,
    load_timestamp
  )
  VALUES (
    src.order_id,
    src.payment_sequential,
    src.payment_type,
    src.payment_installments,
    src.payment_value,
    src.file_name,
    src.file_path,
    current_timestamp()
  )
""")